# Synthefy's 2026 fantasy football rankings with Nori

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Synthefy/synthefy-nori/blob/main/examples/notebooks/nori-fantasy-football-2026.ipynb)

This notebook shows how we used public football data and Nori-6M, Synthefy's tabular AI model, to rank 200 fantasy players for 2026.

We start with information available before each season, test the same method on 2024 and 2025, and then forecast a low, middle, and high points-per-reception (PPR) total for every 2026 player. The middle estimate determines the ranking. All totals cover up to a player's first 15 regular-season appearances through Week 16.

The data comes from [nflverse](https://github.com/nflverse) and the free [Fantasy Football Calculator API](https://fantasyfootballcalculator.com/api-docs).


## Before you run it

The notebook downloads the public data and Nori model when it runs. A Colab GPU is recommended.

To reproduce the published rankings, the notebook uses the same package version and checks that the model and source data still match the September 2 snapshot. If a public source has changed, the notebook stops and identifies the mismatch.


In [ ]:
import importlib.metadata
import subprocess
import sys

REQUIRED_PACKAGES = {
    "synthefy-nori": "0.20.0",
    "nflreadpy": "0.1.5",
}

needs_install = any(
    importlib.metadata.version(package) != wanted
    if package in {dist.metadata["Name"] for dist in importlib.metadata.distributions()}
    else True
    for package, wanted in REQUIRED_PACKAGES.items()
)
if needs_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "synthefy-nori==0.20.0",
            "nflreadpy==0.1.5",
        ]
    )


In [ ]:
import hashlib
import json
import re
import unicodedata
import urllib.request
from collections import defaultdict
from difflib import SequenceMatcher

import nflreadpy as nfl
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.metrics import mean_absolute_error
from synthefy_nori import NoriRegressor

START_SEASON = 2015
HOLDOUT_SEASONS = (2024, 2025)
HOLDOUT_SEASON = HOLDOUT_SEASONS[-1]
FORECAST_SEASON = 2026
TARGET_THROUGH_WEEK = 16
TARGET_GAMES = 15
N_LAGS = 5
ZERO_FILL_LAG_HISTORY = True
POSITIONS = {"QB", "RB", "WR", "TE"}
POSITION_FEATURES = [f"position_{position.lower()}" for position in sorted(POSITIONS)]
MODEL = "nori-6m"

# FFC responses are mutable. These canonical JSON hashes identify the September 2 run.
EXPECTED_FFC_SHA256 = {
    2015: "18345becfb813d5b7c8ecaa09159d7f3028941fbb355b183ce4f5499d4e3ad20",
    2016: "eafc5a14ece4845bbc57e4522daf237f0186159c9fc80b100b5b6052cd9fb090",
    2017: "56caa8abcc4aafe9e7e544f049e0e5ad8cd25bec3aaec2301bcddc93ef506f04",
    2018: "deca10faf3ccfe821f9ac8391c7412f8986ba9e55d2f4b75e2cdcb5ef72d6cb6",
    2019: "b481d4c687c1c7b357e12065c9b6c22f3d1c2542434b18d41167b65305e6e883",
    2020: "59f44898b078d34b7a1dbf0638ef9bd97b4ee39f18d2be06891d32f3c2e8a33d",
    2021: "e4e34fa05ecf61d865753c695d7ffd6ab0adc60019a92e9142a2a572e9058451",
    2022: "6bf1cb8b168b287db32a7f9509d4334eb3a484d1224c871830fee16588e6e1bc",
    2023: "67ac90921bf68e036bcd6c670a20c99e988c872f395cd46f09a65efa3be5078e",
    2024: "c43ad1aa3f5cb3a11df3250182f61561bf13b324224d4d787d90d5810e41bc7e",
    2025: "73dc3a45f98b7bbb20ddfe29fb135be8154fcd2dc5f371c181d934cb1491fc2e",
    2026: "c7101f7c894fde3f8b599556c50fed387ca5810e03d86cdb7b983838e8254e14",
}
EXPECTED_NORI_6M_SHA256 = "a13b2bc31d8db24d17bae6d04844e0adf669e446087b0b7a34c7b05045d61323"

# Updated after the outcome-blind matcher, one-hot position encoding, and Travis Hunter fix.
EXPECTED_HOLDOUT_MAE = {2024: 53.8223823, 2025: 51.2531602}
EXPECTED_FEATURE_TABLE_SHA256 = (
    "4fd642069b13c7aad547ff6a4b6d6bcda788390968727e1efffb2fad3e446f50"
)


## Build the player history

Each row describes one player just before a season begins. It includes preseason average draft position, age, professional experience, draft information, and up to five previous seasons of production and playing time. Players with shorter careers have zeroes for the seasons before they entered the NFL.

Nori predicts each player's PPR point total through Week 16, capped at the player's first 15 regular-season appearances. Every team has played 15 games by then. The player cap prevents a rare traded player from receiving credit for a 16th game caused by different bye weeks.

For every historical test, the row contains only information available before Week 1 of that season. Fantasy Football Calculator supplies the fantasy position, while nflverse supplies the player statistics. This preserves offensive production for dual-role players whose official roster position may be different.


In [ ]:
STAT_COLUMNS = [
    "games",
    "fantasy_points_ppr",
    "attempts",
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "carries",
    "rushing_yards",
    "rushing_tds",
    "targets",
    "receptions",
    "receiving_yards",
    "receiving_tds",
    "target_share",
    "air_yards_share",
    "wopr",
]
ALIASES = {
    "hollywoodbrown": "marquisebrown",
    "gabedavis": "gabrieldavis",
    "chigokonkwo": "chigoziemokonkwo",
    "mitchtrubisky": "mitchelltrubisky",
    "willfuller": "willfullerv",
    "kennygainwell": "kennethgainwell",
}
TEAM_ALIASES = {"LA": "LAR"}


def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def sha256_file(path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def normalize_team(value):
    return TEAM_ALIASES.get(value, value) if isinstance(value, str) else value


def normalize_name(value) -> str:
    if value is None or pd.isna(value):
        return ""
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().lower()
    text = re.sub(r"\b(jr|sr|ii|iii|iv|v)\b", "", text)
    text = re.sub(r"[^a-z0-9]", "", text)
    return ALIASES.get(text, text)


def numeric(value) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


def age_on_september_1(birth_date, season: int) -> float:
    date = pd.to_datetime(birth_date, errors="coerce")
    if pd.isna(date):
        return np.nan
    start = pd.Timestamp(year=season, month=9, day=1)
    return (start - date).days / 365.25


def fetch_ffc(season: int) -> tuple[pd.DataFrame, dict]:
    url = f"https://fantasyfootballcalculator.com/api/v1/adp/ppr?teams=12&year={season}"
    request = urllib.request.Request(url, headers={"User-Agent": "synthefy-nori-notebook/1.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        body = response.read()
    payload = json.loads(body)
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    rows = pd.DataFrame(payload["players"])
    rows["season"] = season
    rows["position"] = rows["position"].replace({"PK": "K"})
    audit = {
        "season": season,
        "url": url,
        "rows": len(rows),
        "sha256": sha256_bytes(canonical),
    }
    return rows.loc[rows["position"].isin(POSITIONS)].copy(), audit


def build_alias_index(players: pd.DataFrame):
    by_exact = defaultdict(set)
    by_name = defaultdict(set)
    by_position = defaultdict(list)
    for row in players.itertuples(index=False):
        if not isinstance(row.gsis_id, str):
            continue
        aliases = {
            normalize_name(getattr(row, column, None))
            for column in ("display_name", "football_name")
        }
        first = getattr(row, "common_first_name", None) or getattr(row, "first_name", None)
        aliases.add(normalize_name(f"{first or ''} {getattr(row, 'last_name', '') or ''}"))
        for alias in aliases - {""}:
            by_exact[(row.position, alias)].add(row.gsis_id)
            by_name[alias].add(row.gsis_id)
            if row.position in POSITIONS:
                by_position[row.position].append((alias, row.gsis_id))
    return by_exact, by_name, by_position


def resolve_candidates(ids, season, team, position, players, stats_index):
    scored = []
    for player_id in ids:
        if player_id not in players.index:
            continue
        metadata = players.loc[player_id]
        rookie = numeric(metadata.get("rookie_season"))
        if not np.isfinite(rookie):
            rookie = numeric(metadata.get("draft_year"))
        prior = stats_index.loc[(season - 1, player_id)] if (season - 1, player_id) in stats_index.index else None
        prior_team_matches = (
            prior is not None
            and normalize_team(prior.get("recent_team")) == normalize_team(team)
        )
        # Use only facts available before the target season. Do not consult
        # same-season stats or the mutable career-ending `last_season` field.
        score = (
            int(np.isfinite(rookie) and rookie <= season),
            int(metadata.get("position") == position),
            int(prior is not None),
            int(prior_team_matches),
            rookie if np.isfinite(rookie) else -np.inf,
        )
        scored.append((score, player_id))
    scored.sort(key=lambda item: (item[0], item[1]), reverse=True)
    if scored and (len(scored) == 1 or scored[0][0] > scored[1][0]):
        return scored[0][1], len(scored)
    return None, len(scored)


def match_player(name, position, season, team, indexes, players, stats_index):
    by_exact, by_name, by_position = indexes
    normalized = normalize_name(name)
    for candidates, method in (
        (by_exact.get((position, normalized), set()), "exact"),
        (by_name.get(normalized, set()), "exact_cross_position"),
    ):
        resolved, candidate_count = resolve_candidates(
            candidates, season, team, position, players, stats_index
        )
        if resolved is not None:
            return resolved, method, candidate_count
    similarity_by_id = {}
    for alias, player_id in by_position[position]:
        score = SequenceMatcher(None, normalized, alias).ratio()
        similarity_by_id[player_id] = max(similarity_by_id.get(player_id, 0.0), score)
    ordered = sorted(similarity_by_id.items(), key=lambda item: item[1], reverse=True)
    if ordered and ordered[0][1] >= 0.90 and (
        len(ordered) == 1 or ordered[0][1] - ordered[1][1] >= 0.025
    ):
        plausible_count = sum(score >= 0.90 for _, score in ordered)
        return ordered[0][0], "fuzzy", plausible_count
    return None, "unmatched", sum(score >= 0.90 for _, score in ordered)


In [ ]:
def first_15_game_targets(seasons: list[int]) -> pd.DataFrame:
    schedules = nfl.load_schedules(seasons).to_pandas()
    regular = schedules[
        (schedules["game_type"] == "REG")
        & (schedules["week"] <= TARGET_THROUGH_WEEK)
    ]
    team_games = pd.concat(
        [
            regular[["season", "game_id", "home_team"]].rename(columns={"home_team": "team"}),
            regular[["season", "game_id", "away_team"]].rename(columns={"away_team": "team"}),
        ],
        ignore_index=True,
    )
    counts = team_games.groupby(["season", "team"])["game_id"].nunique()
    if not (counts == TARGET_GAMES).all():
        raise ValueError(f"Week 16 is not a 15-team-game horizon:\n{counts[counts != TARGET_GAMES]}")

    weekly = nfl.load_player_stats(seasons, summary_level="week").to_pandas()
    weekly = weekly[
        (weekly["season_type"] == "REG")
        & (weekly["week"] <= TARGET_THROUGH_WEEK)
    ]
    weekly = weekly.sort_values(
        ["season", "player_id", "week", "game_id"], kind="stable"
    )
    weekly = weekly.groupby(["season", "player_id"], sort=False).head(TARGET_GAMES)
    targets = (
        weekly.groupby(["season", "player_id"], as_index=False)
        .agg(actual_ppr=("fantasy_points_ppr", "sum"), actual_games=("game_id", "nunique"))
    )
    assert (targets["actual_games"] <= TARGET_GAMES).all()
    return targets


def build_player_seasons():
    stat_seasons = list(range(START_SEASON - N_LAGS, FORECAST_SEASON))
    summary = nfl.load_player_stats(stat_seasons, summary_level="reg").to_pandas()
    summary = summary.sort_values(["season", "player_id"]).drop_duplicates(
        ["season", "player_id"], keep="last"
    )
    stats_index = summary.set_index(["season", "player_id"])

    player_metadata = nfl.load_players().to_pandas()
    player_metadata = player_metadata.drop_duplicates("gsis_id", keep="last")
    players = player_metadata.set_index("gsis_id", drop=False)
    indexes = build_alias_index(player_metadata)

    targets = first_15_game_targets(list(range(START_SEASON, HOLDOUT_SEASON + 1)))
    target_index = targets.set_index(["season", "player_id"])

    rows = []
    source_audit = []
    match_audit = []
    for season in range(START_SEASON, FORECAST_SEASON + 1):
        adp, audit = fetch_ffc(season)
        source_audit.append(audit)
        for player in adp.to_dict(orient="records"):
            player_id, method, candidate_count = match_player(
                player["name"],
                player["position"],
                season,
                player.get("team"),
                indexes,
                players,
                stats_index,
            )
            match_audit.append(
                {
                    "season": season,
                    "name": player["name"],
                    "position": player["position"],
                    "player_id": player_id,
                    "method": method,
                    "candidate_count": candidate_count,
                }
            )
            if player_id is None:
                continue

            metadata = players.loc[player_id]
            draft_year = numeric(metadata.get("draft_year"))
            rookie_season = numeric(metadata.get("rookie_season"))
            career_start = rookie_season if np.isfinite(rookie_season) else draft_year
            row = {
                "season": season,
                "player_id": player_id,
                "player_name": player["name"],
                "position": player["position"],
                "team": normalize_team(player.get("team")),
                "adp": numeric(player.get("adp")),
                "adp_sd": numeric(player.get("stdev")),
                "times_drafted": numeric(player.get("times_drafted")),
                "adp_high": numeric(player.get("high")),
                "adp_low": numeric(player.get("low")),
                "age": age_on_september_1(metadata.get("birth_date"), season),
                "rookie": float(
                    np.isfinite(career_start) and career_start == season
                ),
                "years_since_draft": max(0.0, season - draft_year)
                if np.isfinite(draft_year)
                else np.nan,
                "pro_experience": max(0.0, season - career_start)
                if np.isfinite(career_start)
                else np.nan,
                "draft_round": numeric(metadata.get("draft_round")),
                "draft_pick": numeric(metadata.get("draft_pick")),
                "height": numeric(metadata.get("height")),
                "weight": numeric(metadata.get("weight")),
                "match_method": method,
            }

            target_key = (season, player_id)
            if season <= HOLDOUT_SEASON and target_key in target_index.index:
                target = target_index.loc[target_key]
                row["actual_ppr"] = numeric(target["actual_ppr"])
                row["actual_games"] = numeric(target["actual_games"])
            elif season <= HOLDOUT_SEASON:
                row["actual_ppr"] = 0.0
                row["actual_games"] = 0.0
            else:
                row["actual_ppr"] = np.nan
                row["actual_games"] = np.nan

            for lag in range(1, N_LAGS + 1):
                previous = (
                    stats_index.loc[(season - lag, player_id)]
                    if (season - lag, player_id) in stats_index.index
                    else None
                )
                for column in STAT_COLUMNS:
                    value = numeric(previous[column]) if previous is not None else np.nan
                    if ZERO_FILL_LAG_HISTORY and not np.isfinite(value):
                        value = 0.0
                    row[f"prev{lag}_{column}"] = value
                games = row[f"prev{lag}_games"]
                points = row[f"prev{lag}_fantasy_points_ppr"]
                row[f"prev{lag}_ppg"] = points / games if games > 0 else 0.0
                row[f"prev{lag}_team"] = (
                    normalize_team(previous["recent_team"]) if previous is not None else None
                )

            row["changed_team"] = float(
                isinstance(row["prev1_team"], str)
                and isinstance(row["team"], str)
                and row["prev1_team"] != row["team"]
            )
            row["points_trend"] = (
                row["prev1_fantasy_points_ppr"] - row["prev2_fantasy_points_ppr"]
            )
            row["ppg_trend"] = row["prev1_ppg"] - row["prev2_ppg"]
            rows.append(row)

    frame = pd.DataFrame(rows).sort_values(["season", "adp"]).reset_index(drop=True)
    return frame, pd.DataFrame(source_audit), pd.DataFrame(match_audit), summary


frame, source_audit, match_audit, season_stats = build_player_seasons()

source_audit["expected_sha256"] = source_audit["season"].map(EXPECTED_FFC_SHA256)
source_audit["matches_frozen_source"] = (
    source_audit["sha256"] == source_audit["expected_sha256"]
)
source_mismatches = source_audit.loc[~source_audit["matches_frozen_source"]]
if not source_mismatches.empty:
    raise RuntimeError(
        "A live Fantasy Football Calculator response differs from the September 2 "
        f"publication input:\n{source_mismatches.to_string(index=False)}"
    )

assert frame.loc[frame["season"] == FORECAST_SEASON, "actual_ppr"].isna().all()
assert not frame.duplicated(["season", "player_id"]).any()
assert frame.loc[frame["season"] <= HOLDOUT_SEASON, "actual_games"].le(TARGET_GAMES).all()
assert len(frame.loc[frame["season"] == FORECAST_SEASON]) >= 200

hunter_2025 = frame.loc[
    (frame["season"] == HOLDOUT_SEASON)
    & frame["player_name"].str.fullmatch("Travis Hunter", case=False, na=False)
].iloc[0]
assert hunter_2025["actual_games"] == 7
assert np.isclose(hunter_2025["actual_ppr"], 63.8)

ambiguous_holdout_matches = match_audit.loc[
    match_audit["season"].isin(HOLDOUT_SEASONS)
    & (match_audit["candidate_count"] > 1)
]
assert ambiguous_holdout_matches["player_id"].notna().all()
print(
    {
        "rows_2015_2025": int((frame["season"] < FORECAST_SEASON).sum()),
        "holdout_rows": int((frame["season"] == HOLDOUT_SEASON).sum()),
        "forecast_rows": int((frame["season"] == FORECAST_SEASON).sum()),
        "unmatched": int((match_audit["method"] == "unmatched").sum()),
    }
)
source_audit.tail(3)


## Choose what Nori can see

Every input is available before the season begins. Position is stored as four yes-or-no columns, one each for quarterback, running back, wide receiver, and tight end. Player names, team names, season labels, and final point totals are excluded from the model inputs.

Previous-season statistics are set to zero when a player was not yet in the NFL. Other missing preseason details remain missing so Nori can treat them as unknown.


In [ ]:
BASE_FEATURES = [
    "adp",
    "adp_sd",
    "times_drafted",
    "adp_high",
    "adp_low",
    "age",
    "rookie",
    "years_since_draft",
    "pro_experience",
    "draft_round",
    "draft_pick",
    "height",
    "weight",
]
RECENT_LAG_FEATURES = [
    feature
    for lag in (1, 2)
    for feature in [
        *[f"prev{lag}_{column}" for column in STAT_COLUMNS],
        f"prev{lag}_ppg",
    ]
]
EXTRA_LAG_FEATURES = [
    *[
        f"prev{lag}_{column}"
        for lag in range(3, N_LAGS + 1)
        for column in STAT_COLUMNS
    ],
    *[f"prev{lag}_ppg" for lag in range(3, N_LAGS + 1)],
]
FEATURES = [
    *BASE_FEATURES,
    *RECENT_LAG_FEATURES,
    "changed_team",
    "points_trend",
    "ppg_trend",
    *POSITION_FEATURES,
    *EXTRA_LAG_FEATURES,
]


def prepare_matrix(rows: pd.DataFrame) -> np.ndarray:
    work = rows.copy()
    for position in sorted(POSITIONS):
        work[f"position_{position.lower()}"] = (work["position"] == position).astype(np.float32)
    assert work[POSITION_FEATURES].sum(axis=1).eq(1.0).all()
    return work[FEATURES].to_numpy(dtype=np.float32)


def table_sha256(rows: pd.DataFrame) -> str:
    return sha256_bytes(rows.to_csv(index=False, lineterminator="\n").encode())


assert len(FEATURES) == 105
assert "prev1_fantasy_points_ppr" in FEATURES
assert all(f"prev{lag}_fantasy_points_ppr" in FEATURES for lag in range(1, N_LAGS + 1))
assert not {"season", "player_id", "player_name", "team", "actual_ppr"} & set(FEATURES)

feature_table_hash = table_sha256(frame)
if EXPECTED_FEATURE_TABLE_SHA256 and feature_table_hash != EXPECTED_FEATURE_TABLE_SHA256:
    raise RuntimeError(
        "The assembled nflverse/FFC table differs from the September 2 publication input: "
        f"{feature_table_hash}"
    )

print({"feature_count": len(FEATURES), "feature_table_sha256": feature_table_hash})


## Test the method on 2024 and 2025

Before creating the 2026 list, we replay the process as if we were at the start of 2024 and then 2025. In each test, Nori can use only seasons that had already finished.

We measure accuracy with mean absolute error, or MAE. It is the average number of fantasy points between a forecast and the player's final total, so lower is better. Nori's average miss was 53.82 points in 2024 and 51.25 points in 2025.

Nori produces low, middle, and high estimates. The code calls these P10, P50, and P90. P50 is the median: half of Nori's possible outcomes are higher and half are lower. It is the number used for the rankings and MAE.


In [ ]:
checkpoint_path = hf_hub_download(repo_id="Synthefy/Nori", filename="nori.pt")
checkpoint_hash = sha256_file(checkpoint_path)
assert checkpoint_hash == EXPECTED_NORI_6M_SHA256, (
    "The public Nori-6M checkpoint changed; review before comparing results."
)


def run_holdout(season: int):
    train = frame.loc[frame["season"] < season].copy()
    holdout = frame.loc[frame["season"] == season].copy()
    estimator = NoriRegressor(model_path=checkpoint_path)
    estimator.fit(
        prepare_matrix(train),
        train["actual_ppr"].to_numpy(dtype=np.float64),
    )
    X_holdout = prepare_matrix(holdout)
    p10, p50, p90 = estimator.predict(
        X_holdout,
        output_type="quantiles",
        quantiles=[0.1, 0.5, 0.9],
    )
    memory_report = getattr(estimator, "memory_report_", None)
    dropped_context_rows = (
        0 if memory_report is None else int(memory_report.get("dropped_context_rows", 0))
    )
    assert dropped_context_rows == 0, memory_report
    actual = holdout["actual_ppr"].to_numpy(dtype=np.float64)
    mae = mean_absolute_error(actual, p50)
    coverage = np.mean((actual >= p10) & (actual <= p90))
    expected = EXPECTED_HOLDOUT_MAE[season]
    if expected is not None:
        np.testing.assert_allclose(mae, expected, rtol=0.0, atol=0.05)
    predictions = holdout[
        ["season", "player_id", "player_name", "position", "team", "actual_ppr", "actual_games"]
    ].copy()
    predictions["p10"] = p10
    predictions["p50"] = p50
    predictions["p90"] = p90
    predictions["absolute_error"] = np.abs(actual - p50)
    return {
        "season": season,
        "rows": len(holdout),
        "mae": mae,
        "coverage": coverage,
        "dropped_context_rows": dropped_context_rows,
    }, predictions


holdout_metrics = []
holdout_predictions = {}
for holdout_season in HOLDOUT_SEASONS:
    metrics, predictions = run_holdout(holdout_season)
    holdout_metrics.append(metrics)
    holdout_predictions[holdout_season] = predictions

holdout_metrics = pd.DataFrame(holdout_metrics)
holdout_metrics.round(3)


## Create the 2026 rankings

For the final run, Nori uses the completed 2015–2025 seasons to forecast every player in the September 2, 2026 preseason snapshot. Players are ordered by the median PPR forecast alone.


In [ ]:
context_2026 = frame.loc[frame["season"] < FORECAST_SEASON].copy()
query_2026 = frame.loc[frame["season"] == FORECAST_SEASON].copy()

nori_2026 = NoriRegressor(model_path=checkpoint_path)
nori_2026.fit(
    prepare_matrix(context_2026),
    context_2026["actual_ppr"].to_numpy(dtype=np.float64),
)
X_query_2026 = prepare_matrix(query_2026)
p10_2026, p50_2026, p90_2026 = nori_2026.predict(
    X_query_2026,
    output_type="quantiles",
    quantiles=[0.1, 0.5, 0.9],
)

memory_report_2026 = getattr(nori_2026, "memory_report_", None)
dropped_context_rows_2026 = (
    0
    if memory_report_2026 is None
    else int(memory_report_2026.get("dropped_context_rows", 0))
)
assert dropped_context_rows_2026 == 0, memory_report_2026

predictions_2026 = query_2026[
    ["player_id", "player_name", "position", "team", "adp"]
].copy()
predictions_2026["p10"] = p10_2026
predictions_2026["p50"] = p50_2026
predictions_2026["p90"] = p90_2026

board_2026 = (
    predictions_2026.sort_values(
        ["p50", "adp", "player_name"],
        ascending=[False, True, True],
        kind="stable",
    )
    .head(200)
    .reset_index(drop=True)
)
board_2026.insert(0, "rank", np.arange(1, len(board_2026) + 1))
assert len(board_2026) == 200
assert board_2026["p50"].is_monotonic_decreasing
assert (board_2026["p10"] <= board_2026["p50"]).all()
assert (board_2026["p50"] <= board_2026["p90"]).all()

board_2026.head(20).round({"adp": 1, "p10": 1, "p50": 1, "p90": 1})


In [ ]:
# Optional export for a draft sheet or website table.
board_2026.to_csv("synthefy_nori_2026_fantasy_top_200.csv", index=False)
print("Wrote synthefy_nori_2026_fantasy_top_200.csv")


## How to read the rankings

The median point forecast determines the order. The low and high estimates show how wide the range of plausible outcomes is for each player.

This is a September 2 preseason snapshot. Check current injury and depth-chart news before using it for a draft.

nflverse player data is licensed under CC BY 4.0, and Fantasy Football Calculator requests attribution for its free API.

Players with fewer than five professional seasons have zeroes for the years before they entered the NFL. Keep that convention in mind when reading forecasts for rookies, younger players, and players returning after long absences.
